In [1]:
#check on December-8, 2025 by Hengcong Liu

import pandas as pd
import numpy as np
import statsmodels.api as sm
from pathlib import Path
from sklearn.preprocessing import StandardScaler

# ================================================================
#                       1. 路径设置
# ================================================================
risk_file = Path('output/1_risk_factor/risk_factor.xlsx')
all_data_file = Path('../1_clean/data_merge_all_20260110.xlsx')

output_dir = Path('output/2_weight')
output_dir.mkdir(parents=True, exist_ok=True)

coef_file = output_dir / 'logistic_coef_pval.xlsx'
weight_file = output_dir / 'weight.xlsx'

# ================================================================
#                       2. 数据读取
# ================================================================
risk_factors = pd.read_excel(risk_file)
all_data = pd.read_excel(all_data_file)
all_data = all_data[~(all_data['host_species'].isna() & all_data['standard virus name'].isna())].reset_index(drop=True)
print(all_data.shape[0])

# ================================================================
#          3. 构建县区调查状态（避免重复 union）
# ================================================================
county_surveyed = pd.DataFrame({
    'gb': all_data['gb'].unique(),
    'status': 1
})

# ================================================================
#                       4. 左连接 + 填补
# ================================================================
data = (
    risk_factors
    .merge(county_surveyed, on='gb', how='left')
    .assign(status=lambda df: df['status'].fillna(0).astype(int))
)

# ================================================================
#                     5. X / y 构建 + 标准化
# ================================================================
X = data.drop(columns=['gb', 'status'])
y = data['status']

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
X_const = sm.add_constant(X_scaled)

# ================================================================
#           6. Logistic 回归（添加 penalization 防止崩溃）
# ================================================================
logit_model = sm.Logit(y, X_const)
result = logit_model.fit(disp=False) # dont show the interation information, dont change the results
print(result.summary())

# === 7. 输出 coef 和 p-value ===
coef_pval_df = pd.DataFrame({
    'Variable': result.params.index,
    'Coefficient': result.params.values.round(4),
    'p_value': result.pvalues.values.round(4)
})
coef_pval_df.to_excel(coef_file, index=False)
print(f"回归系数和p-value已保存到: {coef_file}")

# ================================================================
#        8. 计算预测概率（clip 避免出现 prob=0 导致权重爆炸）
# ================================================================
data['prob'] = result.predict(X_const)

# ================================================================
#                   9. 生成调查县区权重
# ================================================================
survey = data.loc[data['status'] == 1, ['gb', 'prob']].copy()

survey['recip'] = 1 / survey['prob']
survey['rescaled'] = survey['recip'] / survey['recip'].mean()

survey.to_excel(weight_file, index=False)

print(f"✔ 权重数据已保存到: {weight_file}")

47196
                           Logit Regression Results                           
Dep. Variable:                 status   No. Observations:                 2891
Model:                          Logit   Df Residuals:                     2870
Method:                           MLE   Df Model:                           20
Date:                Sat, 10 Jan 2026   Pseudo R-squ.:                 0.06035
Time:                        13:36:18   Log-Likelihood:                -1516.4
converged:                       True   LL-Null:                       -1613.8
Covariance Type:            nonrobust   LLR p-value:                 1.200e-30
                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
const                      1.2328      0.048     25.632      0.000       1.139       1.327
Bio3                      -0.1916      0.136     -1.408      0.159      -0.458       0.07